# A06 · RTB协议角色、底价与截尾反馈

先修：公用31–34；A05。

证据类型：**明确构造机制实验（非真实广告日志）**。本章不声称工业系统复现或线上收益。

学习顺序：先预测 → 只改一个变量 → 运行 → 解释结果。

## 问题与输入输出契约

此处是协议角色的最小状态机，而非OpenRTB实现。DSP提交报价；ADX在媒体底价reserve和竞争价下决定成交。二价支付max(reserve, competitor)，一价支付自己的报价。报价达到公开底价但仍输标时，不知道完整竞争价，只知道超过自己报价的下界（此例无并列竞争）；未达到底价时则不能推出竞争价下界；超时不提供这一价格信息。竞价对手不是媒体底价。

运行前预测：本章只改变的变量会使哪一项结果变化？哪些不变量必须保持？

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## 运行单变量对照

输入是源码中可见的小型构造数据，输出单位和约束按上文定义。运行断言核验数学与账本。

In [2]:
rows=[ac.rtb(),ac.rtb(reserve=1.3),ac.rtb(mechanism='first'),ac.rtb(timeout=True),ac.rtb(competitor=1.5)]
display(pd.DataFrame(rows).drop(columns='stages'))
assert rows[0]['payment']==1 and rows[1]['payment']==0
assert rows[1]['observed_market_price'] is None
assert rows[3]['lost_price_lower_bound'] is None
assert rows[4]['lost_price_lower_bound']==1.2
assert rows[4]['observed_market_price'] is None

,win,payment,observed_market_price,lost_price_lower_bound
0,True,1.0,1.0,NaN
1,False,0.0,NaN,NaN
2,True,1.2,1.0,NaN
3,False,0.0,NaN,NaN
4,False,0.0,NaN,1.2


## 解释与局限

只提高reserve可以使原本赢标的请求不成交；一价机制下同样报价有不同支付。不能把只观察到赢标市场价的样本当作完整市场分布，超时也不等于价格输标。

## 查看源码，再做练习

给状态机增加无效币种、物料不合规和重复notice；区分no-bid、timeout、loss和不展示。

In [3]:
import inspect
print(inspect.getsource(ac.rtb))

def rtb(bid=1.2, competitor=1., reserve=.8, mechanism='second', timeout=False):
    if mechanism not in ('first','second'): raise ValueError('Invalid mechanism')
    win=not timeout and bid>=max(competitor,reserve)
    return dict(win=win, payment=(bid if mechanism=='first' else max(reserve,competitor)) if win else 0.,
                observed_market_price=competitor if win else None,
                lost_price_lower_bound=bid if not win and not timeout and bid>=reserve else None,
                stages=['SSP request','ADX eligibility','DSP response','auction','win/loss notice','feedback'])



## 来源与下一步

A-CCR 27–31页；A-RTB 18–19、38页。这是Lab自行设计的小实验，不复制讲义全文。

[GitHub中文单章](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [作者RTB讲义](https://wnzhang.net/slides/cikm-16-rtb-full.pdf)。中文PPT不是现代广告工程全套课；不将历史平台机制描述当作当前市场事实。

完成练习后回到广告学习路径，继续下一语义单元。